# 03. Range sums

**Tier:** Deep. **Why you'd use this:** The product of every element except `nums[i]` is the product of everything before `i` times the product of everything after `i`. Walk left to right for the prefixes.

**After this lesson you can:** answer many range-sum queries from one prefix array; subtract two prefix values.
About 20 minutes. **Uses:** [Running sum](02-running-sum-lesson.ipynb)


## Quick review

1. How do you get nums[L] + … + nums[R] from a prefix array?
2. Why precompute prefixes when you have many range queries?

<details><summary>Answers</summary>

For 03 range sum lesson: skim [Running sum](02-running-sum-lesson.ipynb) if needed.
</details>


## Recognize it

You need the sum of many slices `nums[left:right+1]` after one preprocess. The prompt says "range sum", "sum between indexes", or "answer queries in constant time after a build".

Signals: repeated contiguous sums, and a willingness to spend linear memory up front. A look-alike that is not this pattern: product of all elements except self — that uses prefix and suffix products, not range sums with subtraction.

A second look-alike: one sliding window of fixed length. That also asks for contiguous sums, but the windows move by one and share almost everything with the previous window. Prefer the slide when that is the only query shape; prefer a prefix table when left and right jump around.


## Why it works this way

The sum from index `L` through `R` equals the sum of the first `R + 1` values minus the sum of the first `L` values. If `prefix[0] = 0` and `prefix[i]` stores the sum of `nums[0:i]`, then the range is `prefix[R + 1] - prefix[L]`. Building `prefix` costs one linear pass. Each query after that is two reads and a subtract — constant time.

Answering each query with `sum(nums[L:R+1])` rescans that slice. Many queries on large slices push the total work toward `n` times the number of queries. The prefix table moves that cost to a single preprocess. Memory grows with `n` for the table. That trade is the design reason timed challenges ask for prefix sums: pay once, answer often, and keep each later check cheap under a busy query load.

Off-by-one errors dominate bugs in this pattern. Inclusive `R` needs `prefix[R + 1]`, not `prefix[R]`. An empty left side uses `prefix[0]`, which is why the leading zero exists. Without it you special-case `L == 0` on every query and the special case becomes another place to slip.

Products use a similar shape with multiplication and a leading `1`, but zeros break naive division shortcuts — that is why the product-except-self practice avoids division. Keep sum prefixes and product prefixes as cousin patterns, not the same code. Sliding windows still win when you only need equal-length blocks moving by one and you do not want the extra array sitting in memory.

Mechanically the exclusive layout is the clean contract: `prefix[i]` never includes `nums[i]`, so a half-open mental model `[L, R+1)` maps straight onto two table reads. Inclusive running-sum arrays from the previous lesson are one shift away; convert once at build time rather than inventing a new formula per query. Updates that change a single `nums[i]` after the table is built force a rebuild or a heavier structure — plain prefixes assume a static array.


## When to use it

Use a prefix table when many contiguous range sums (or average queries) share one array. Use a sliding window when windows move by one and length is fixed or constrained. Use a segment tree or fenwick structure only when updates interleave with queries — that is beyond this lesson. Prefer a direct `sum` slice for one-off questions on small lists.

If queries ask for range minimums or gcds instead of sums, subtraction of prefixes does not work. Those need sparse tables or segment trees. Keep the sum identity for sums alone.


## Brute force, and why it's slow

Answer each query by scanning the slice. Count additions across several queries.


In [1]:
def range_sum_slow(nums, queries):
    additions = 0
    answers = []
    for left, right in queries:
        total = 0
        for index in range(left, right + 1):
            total += nums[index]
            additions += 1
        answers.append(total)
    return answers, additions


nums = list(range(1, 21))
queries = [(0, 19), (2, 10), (5, 15), (0, 5)]
print(range_sum_slow(nums, queries))

([210, 63, 121, 21], 46)


## Key insight

`sum(nums[L:R+1]) = prefix[R+1] - prefix[L]` when `prefix[i]` is the sum of the first `i` values and `prefix[0] = 0`. Build once; answer each query with two reads.


## Worked example

Build one prefix array, then return the sum of `nums[left]` through `nums[right]`, inclusive.


In [2]:
def range_sum(nums: list[int], left: int, right: int) -> int:
    # 1. prefix[i] = sum of the first i values; prefix[0] = 0.
    prefix = [0]
    for value in nums:
        prefix.append(prefix[-1] + value)
    # 2. Inclusive range via two lookups.
    return prefix[right + 1] - prefix[left]


print(range_sum([1, 2, 3, 4], 1, 3))
print(range_sum([1, 2, 3, 4], 0, 0))
print(range_sum([2, -1, 3], 0, 2))

9
1
4


## Trace

`nums = [1, 2, 3, 4]` → `prefix = [0, 1, 3, 6, 10]`

| left | right | `prefix[right+1]` | `prefix[left]` | sum |
|---|---|---|---|---|
| 1 | 3 | 10 | 1 | **9** |
| 0 | 0 | 1 | 0 | **1** |


## Cost

Build is proportional to `n`. Each query is constant time. Extra memory is `n + 1` prefix slots. Scanning each query costs work proportional to the slice length, every time. With `q` large queries that difference is the whole point of the preprocess.


## Variants

Not solved here: [Product of Array Except Self](../../challenges/problems/06-product-of-array-except-self.ipynb), [Subarray Sum Equals K](../../challenges/problems/43-subarray-sum-equals-k.ipynb), [Maximum Subarray](../../challenges/problems/21-maximum-subarray.ipynb).


## Common mistake

This bug is silent. The code returns `prefix[right] - prefix[left]`, dropping the value at `right` from an inclusive range. There is no traceback. The print is `5`, and the answer is `9`.


In [3]:
def range_sum_wrong(nums, left, right):
    prefix = [0]
    for value in nums:
        prefix.append(prefix[-1] + value)
    return prefix[right] - prefix[left]  # exclusive on the right by mistake


print(range_sum_wrong([1, 2, 3, 4], 1, 3))

5


The fix uses `prefix[right + 1]` for an inclusive right index. The worked example does that, so the same query prints `9`.


In [4]:
print(range_sum([1, 2, 3, 4], 1, 3))

9


## Python idioms

Build with a loop, or `prefix = [0, *accumulate(nums)]`. Keep the leading `0` so the `L == 0` case needs no branch.


In [5]:
from itertools import accumulate


def range_sum_lib(nums: list[int], left: int, right: int) -> int:
    prefix = [0, *accumulate(nums)]
    return prefix[right + 1] - prefix[left]


print(range_sum_lib([1, 2, 3, 4], 1, 3))

9


## Recap

- A prefix table turns a range sum into two lookups.
- Inclusive right uses `prefix[right + 1]`.
- Pay linear memory once to answer many queries fast.

Next: [practice](03-range-sum-practice.ipynb).
